# 方向一 Colab Launcher：计算病理与先验引导的多模态生存预测（对应 Paper 1）

本 Notebook 专为 **Google Colab Pro (`L4 24GB` 首选)** 设计，遵循**“Colab 本地 SSD (`/content/`) 高速运算 + Google Drive 持久化日志与轻量级产出”**策略：
- **不会**把 20–40 GB 原始 `.svs` 塞满 Google Drive，只在 `/content/` 临时高速盘切块提特征；
- **自动**将所有命令的完整日志实时同步至 `MyDrive/MedDemo_backup/direction1/logs/`（配合本机 Google Drive 桌面端，本地 Agent 可直接读取报错进行 Debug）；
- **自动**将 `clinical.csv`、`splits.csv`、`slide_priors.csv`、TRIDENT 提取的 `.h5` 特征包与最终图表报告备份至 Google Drive，断连重开无需重提特征。

## Step 1：硬件与磁盘检查 + 挂载 Google Drive

In [ ]:
!nvidia-smi
!df -h /content

from google.colab import drive
drive.mount('/content/drive')

## Step 2：配置仓库来源与日志/备份路径
- 若代码在你的 **GitHub 仓库**（推荐，便于本地 Agent 修复后直接 `git pull`）：填写 `GITHUB_REPO_URL`。
- 若你直接把代码文件夹上传到了 Google Drive：把 `GITHUB_REPO_URL` 留空，并设置 `DRIVE_CODE_DIR`。

In [ ]:
import os
import subprocess
from pathlib import Path

# === 请按需修改以下参数 ===
GITHUB_REPO_URL = "https://github.com/WilliamWu9871/MedDemo.git"  # 私有库请改用带 PAT 的 URL
GIT_BRANCH = "main"
DRIVE_CODE_DIR = "/content/drive/MyDrive/MedDemo"  # 仅当 GITHUB_REPO_URL 为空时使用

WORK_ROOT = Path("/content/MedDemo")
DIR1_ROOT = WORK_ROOT / "direction1_pathology_multimodal_survival"
VENV_DIR = Path("/content/venv_meddemo_py311")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/MedDemo_backup/direction1")
DRIVE_LOG_DIR = DRIVE_BACKUP_DIR / "logs"
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOG_DIR.mkdir(parents=True, exist_ok=True)

# 从 GitHub 拉取或从 Drive 同步源码到高速本地盘 /content/MedDemo
if GITHUB_REPO_URL:
    if (WORK_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(WORK_ROOT), "pull", "origin", GIT_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "-b", GIT_BRANCH, GITHUB_REPO_URL, str(WORK_ROOT)], check=True)
else:
    if not Path(DRIVE_CODE_DIR).exists():
        raise FileNotFoundError(f"请填写 GITHUB_REPO_URL 或将代码放到 {DRIVE_CODE_DIR}")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-av", "--exclude", "*.pdf", f"{DRIVE_CODE_DIR}/", f"{WORK_ROOT}/"], check=True)

os.chdir(DIR1_ROOT)
print("Working dir:", os.getcwd())
print("Live logs dir (synced to Drive):", DRIVE_LOG_DIR)
print("Isolated Python 3.11 venv:", VENV_DIR)


def run_logged(cmd: str, log_name: str, cwd: Path = DIR1_ROOT, use_venv: bool = True) -> None:
    """Run shell command, stream output live, and flush full stdout/stderr to Google Drive."""
    log_path = DRIVE_LOG_DIR / log_name
    activate_script = VENV_DIR / "bin" / "activate"
    full_cmd = f"source '{activate_script}' && {cmd}" if (use_venv and activate_script.exists()) else cmd
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    print(f"==> Running: {cmd}\n==> Log file: {log_path}")
    with log_path.open("w", encoding="utf-8") as log_f:
        proc = subprocess.Popen(
            ["bash", "-lc", full_cmd],
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_f.write(line)
            log_f.flush()
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed (exit={ret}). Full traceback saved at: {log_path}")


def backup_to_drive(project_tag: str = "tcga_lihc") -> None:
    """Sync lightweight metadata, splits, TRIDENT .h5 features, and reports to Drive (excludes raw .svs)."""
    for rel in ["splits", f"data/{project_tag}/metadata", f"data/{project_tag}/thumbnails", f"outputs/{project_tag}"]:
        src = DIR1_ROOT / rel
        if src.exists():
            dst = DRIVE_BACKUP_DIR / rel
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--exclude", "*.svs", f"{src}/", f"{dst}/"], check=True)
    print(f"[Backup] Synced lightweight outputs/features -> {DRIVE_BACKUP_DIR}")


def restore_from_drive(project_tag: str = "tcga_lihc") -> None:
    """Restore cached features/splits/outputs from Drive into /content/ after a Colab reconnect."""
    for rel in ["splits", f"data/{project_tag}/metadata", f"data/{project_tag}/thumbnails", f"outputs/{project_tag}"]:
        src = DRIVE_BACKUP_DIR / rel
        if src.exists():
            dst = DIR1_ROOT / rel
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", f"{src}/", f"{dst}/"], check=True)
    print(f"[Restore] Restored cached artifacts from {DRIVE_BACKUP_DIR}")

## Step 3：创建 Python 3.11 隔离环境、安装依赖并拉取 `TRIDENT` & `SurvPath`
Colab 默认 Python 3.13 与 `TRIDENT`（要求 `>=3.10,<3.13`）不兼容，因此本步使用 `uv` 自动创建 `/content/venv_meddemo_py311` 隔离环境并安装 CUDA 版 PyTorch、`openslide`/`libvips`、项目依赖及 `TRIDENT` & `SurvPath`（后续步骤将自动激活该环境）。首次运行约耗时 2–4 分钟；每次 Colab 会话重建时需要重新执行一次。

In [ ]:
run_logged(
    "set -euo pipefail && "
    "apt-get update -qq && apt-get install -y -qq openslide-tools libvips && "
    "(command -v uv >/dev/null 2>&1 || pip install -q uv) && "
    f"uv venv --clear --seed --python 3.11 '{VENV_DIR}' && "
    f"source '{VENV_DIR}/bin/activate' && "
    "python --version && "
    f"uv pip install --python '{VENV_DIR}/bin/python' "
    "torch torchvision lifelines scikit-survival 'scikit-learn>=1.4' statsmodels patsy shap "
    "h5py 'numpy>=1.24' pandas pyarrow scipy matplotlib seaborn tqdm requests huggingface_hub "
    "openslide-python openslide-bin pyyaml Pillow opencv-python-headless && "
    "bash scripts/setup_repos.sh --with-patch-encoders && "
    "python -c \"import sys, torch, trident; print('OK | Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| TRIDENT:', trident.__file__)\"",
    "00_setup_env.log",
    use_venv=False,
)
run_logged("python -m unittest discover -s ../tests -v", "00_unit_tests.log")

## Step 4A：合成数据端到端闭环验收（`--synthetic`，约 1 分钟）
先验证所有代码管线、防泄漏划分、5 折消融、RCS 非线性检验与热力图叠加在当前 Colab 运行时完全畅通。

In [ ]:
run_logged(
    "python scripts/00_download_tcga_subset.py --synthetic --n-patients 72 && "
    "python scripts/01_create_patient_splits.py --synthetic && "
    "python scripts/03_compute_pseudo_prior_tsr.py --synthetic && "
    "python scripts/04_train_ablation.py --synthetic --epochs 15 && "
    "python scripts/05_evaluate_and_explain.py --synthetic",
    "01_synthetic_e2e.log",
)
backup_to_drive("synthetic")

In [ ]:
# 预览合成数据验收报告与关键图表
from IPython.display import Image, Markdown, display

rep_dir = DIR1_ROOT / "outputs/synthetic/report"
display(Markdown((rep_dir / "ablation_cindex_table.md").read_text(encoding="utf-8")))
for fig_name in ["loss_curves.png", "km_survival_curves.png", "rcs_tsr_curve.png", "multivariable_cox_forest.png"]:
    p = rep_dir / fig_name
    if p.exists():
        display(Image(filename=str(p)))
heatmaps = sorted((rep_dir / "heatmaps").glob("*_overlay.png"))
if heatmaps:
    display(Image(filename=str(heatmaps[0])))

## Step 4B：真实 TCGA-LIHC / TCGA-GBM 实验流水线
1. 默认骨干 **`hibou_l`**（HistAI `histai/hibou-L`，Apache 2.0，1024 维，`224×224 @ 20x`）在 Hugging Face 上启用了自动门控（`gated=auto`）：运行 Stage 2 前请访问 https://huggingface.co/histai/hibou-L 点击一次 **Agree** 并在下方填写 `HF_TOKEN`；若无 Token 可临时将 `PATCH_ENCODER` 改为 `"resnet50"`（`PATCH_SIZE = 256`）跑通全流程。
2. 若 Colab 之前断开过且 Drive 已有备份，先取消注释执行 `restore_from_drive(PROJECT_TAG)` 即可跳过下载与提特征。

In [ ]:
CONFIG_FILE = "configs/tcga_lihc_demo.yaml"   # 或改为 configs/tcga_gbm_demo.yaml
PROJECT_TAG = "tcga_lihc"                     # 对应 configs 中的目录名 (tcga_lihc / tcga_gbm)
N_PATIENTS = 70
PATCH_ENCODER = "hibou_l"                     # 默认使用 HistAI Hibou-L (DINOv2 ViT-L/14, 1024d)
PATCH_SIZE = 224                              # hibou_l 使用 224；conch_v15/resnet50 可设为 256
HF_TOKEN = ""                                 # 请先在 https://huggingface.co/histai/hibou-L 点击 Agree 并填入 HF Token

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)

# 如为断线重连恢复阶段，取消下行注释从 Drive 恢复已提取的特征：
# restore_from_drive(PROJECT_TAG)

In [ ]:
# Stage 0 & Stage 1：GDC 拉取 60-80 例 Open Access 诊断级切片 (-DX) + 患者级 5 折划分
run_logged(
    f"python scripts/00_download_tcga_subset.py --config {CONFIG_FILE} --n-patients {N_PATIENTS} --download && "
    f"python scripts/01_create_patient_splits.py --config {CONFIG_FILE}",
    f"02_{PROJECT_TAG}_stage0_1.log",
)
backup_to_drive(PROJECT_TAG)

In [ ]:
# Stage 2：TRIDENT 前景组织分割、20x 切块与特征提取（在 /content/ 本地高速盘运行）
run_logged(
    f"bash scripts/02_extract_trident_features.sh "
    f"--wsi-dir data/{PROJECT_TAG}/svs "
    f"--job-dir outputs/{PROJECT_TAG}/trident_feats "
    f"--patch-encoder {PATCH_ENCODER} --mag 20 --patch-size {PATCH_SIZE}",
    f"03_{PROJECT_TAG}_trident.log",
)
# 提取完成后立即把轻量的 .h5 特征包同步回 Google Drive（原始 .svs 不备份）
backup_to_drive(PROJECT_TAG)

In [ ]:
# Stage 3 & Stage 4：Kather-100K 探针 pTSR + 6 模型消融训练 + 临床统计与热力图验收
feats_candidates = list((DIR1_ROOT / f"outputs/{PROJECT_TAG}/trident_feats").rglob(f"features_{PATCH_ENCODER}"))
if not feats_candidates:
    raise FileNotFoundError(
        f"未找到 features_{PATCH_ENCODER} 目录（位于 outputs/{PROJECT_TAG}/trident_feats 下），"
        f"请先运行上方 Stage 2 单元格提取 {PATCH_ENCODER} 特征。"
    )
feats_dir = sorted(feats_candidates)[-1]
print("Detected TRIDENT features dir:", feats_dir)

run_logged(
    f"python scripts/03_compute_pseudo_prior_tsr.py --config {CONFIG_FILE} --feats-dir '{feats_dir}' --patch-encoder {PATCH_ENCODER} && "
    f"python scripts/04_train_ablation.py --config {CONFIG_FILE} --feats-dir '{feats_dir}' --all-seeds && "
    f"python scripts/05_evaluate_and_explain.py --config {CONFIG_FILE}",
    f"04_{PROJECT_TAG}_train_eval.log",
)
backup_to_drive(PROJECT_TAG)

In [ ]:
# 预览真实队列最终报告与热力图
from IPython.display import Image, Markdown, display

real_rep = DIR1_ROOT / f"outputs/{PROJECT_TAG}/report"
if (real_rep / "ablation_cindex_table.md").exists():
    display(Markdown((real_rep / "ablation_cindex_table.md").read_text(encoding="utf-8")))
for fig_name in ["loss_curves.png", "km_survival_curves.png", "rcs_tsr_curve.png", "multivariable_cox_forest.png", "shap_summary.png"]:
    p = real_rep / fig_name
    if p.exists():
        display(Image(filename=str(p)))
for hm in sorted((real_rep / "heatmaps").glob("*_overlay.png"))[:3]:
    display(Image(filename=str(hm)))